In [ ]:
import sys
print(sys.executable)
print(sys.prefix)

# Just ensure you are using the right venv helps with cross contributor portability.
# the expected output should be something like 
# /Users/rishavbhagat/developer/ChartQA-IIITH/.venv/bin/python
# /Users/rishavbhagat/developer/ChartQA-IIITH/.venv

/Users/rishavbhagat/developer/ChartQA-IIITH/.venv/bin/python
/Users/rishavbhagat/developer/ChartQA-IIITH/.venv


# The begining of the basic data set exploration

Herin we start by having a look at the testing data only, 


In [ ]:
PATHS = ["../datasets/HuggingFaceM4/ChartQA/data/train-00000-of-00003-49492f364babfa44.parquet", 
        "../datasets/HuggingFaceM4/ChartQA/data/train-00001-of-00003-7302bae5e425bbc7.parquet",
        "../datasets/HuggingFaceM4/ChartQA/data/train-00002-of-00003-194c9400785577a2.parquet" 
        ]

In [ ]:
# import os
# import sys

# # 1. Print the location of the Python interpreter executable
# print("Python Executable:", sys.executable)

# # 2. Print the current working directory (where Python looks for relative paths)
# print("Current Working Directory:", os.getcwd())

# # 3. List files in the current working directory to see what is visible
# print("Files in current directory:", os.listdir("../datasets/HuggingFaceM4/ChartQA/data"))


In [ ]:
import pandas as pd

# Load only the first few rows to inspect the structure

df_preview = pd.read_parquet(PATHS[0], engine="pyarrow").head()
print(df_preview.info())
print(df_preview.columns)


In [ ]:
import io
import matplotlib.pyplot as plt
from PIL import Image

def decode_chart_image(row, image_col='image'):
    val = row[image_col]
    if isinstance(val, dict) and 'bytes' in val:
        return Image.open(io.BytesIO(val['bytes']))
    elif isinstance(val, bytes):
        return Image.open(io.BytesIO(val))
    elif isinstance(val, str):
        return Image.open(val)
    else:
        raise ValueError("Unsupported image column format. Ensure it contains bytes or paths.")




In [7]:
# Select 4 samples to inspect closely
sample_df = df_preview.head(40)

# Set up a grid layout (2 rows, 2 columns) with a dark facecolor
fig, axes = plt.subplots(2, 2, figsize=(14, 12), facecolor='black')
axes = axes.flatten()

for i, (_, row) in enumerate(sample_df.iterrows()):
    ax = axes[i -1]
    ax.set_facecolor('black')
    
    # Load and display chart image
    img = decode_chart_image(row)
    ax.imshow(img)
    ax.axis('off')
    
    # Extract query text and target answer label safely
    # Adjust string keys if your columns are named 'question' or 'answer'
    query_text = row.get('query', row.get('question', 'No Query Found'))
    label_text = row.get('label', row.get('answer', 'No Label Found'))
    
    # Wrap long questions so they fit nicely as titles
    wrapped_query = "\n".join([query_text[j:j+50] for j in range(0, len(query_text), 50)])
    
    # Title display (Cyan for query text, Green for ground truth label)
    ax.set_title(
        f"Q: {wrapped_query}\n\nAns: {label_text}", 
        color='#00ffff', 
        fontsize=10, 
        loc='left', 
        pad=10,
        fontweight='bold'
    )

plt.tight_layout()
plt.subplots_adjust(hspace=0.4) # Add space for multi-line text titles
plt.show()


NameError: name 'df_preview' is not defined

In [ ]:
# checking the sort of questions and answers being asked

# Simple LLM run to categorise each query into certain buckets, 1. Quantitative, 2. NUmerical, 3. Reasoning, and similar for the answers, note we will do a shuffled sampling since the Population is arounf 32000, a sample of size 10 sampled 10 times randomly should work? or u suggest

In [ ]:
from pathlib import Path

print(Path.cwd().parent/".env")

In [ ]:


sample_size = 350
IMG_COLS = ["image"]  # change to your image column name(s)

df_sample_train = []
for path in PATHS[:3]:
    df = pd.read_parquet(path, engine="pyarrow")
    df_sample_train.append(df.sample(n=sample_size, random_state=42).copy())

df_sample_train_0, df_sample_train_1, df_sample_train_2 = df_sample_train

# Show the data without the image columns
for i, d in enumerate(df_sample_train):
    print(f"--- sample {i}: {d.shape} ---")
    display(d.drop(columns=IMG_COLS, errors="ignore").head())

In [ ]:
# import os
# import json
# import pandas as pd
# from tqdm import tqdm
# from pathlib import Path
# from litellm import completion
# from dotenv import load_dotenv

# load_dotenv(dotenv_path=Path.cwd().parent / ".env")
# if not os.getenv("DEEPSEEK_API_KEY"):
#     raise ValueError("DEEPSEEK_API_KEY not found in .env file or environment.")

# system_prompt = """
# You are a data evaluation assistant analyzing a ChartQA dataset. 
# Classify the given Chart Question (Query) and Answer into exactly one of these defined buckets.

# Query Categories:
# - "Quantitative": Asking for direct chart data extraction (e.g., "What is the value of X?").
# - "Numerical": Mathematical operations required (e.g., "Sum of X and Y", "Average growth").
# - "Reasoning": Requires logical synthesis, trends, or comparisons (e.g., "Which year performed worst?").

# Answer Categories:
# - "Pure Numerical": Just numbers (e.g., "45", "12.5").
# - "Alphanumeric/Units": Numbers containing metrics (e.g., "45%", "$12M").
# - "Descriptive Text": Pure string phrases (e.g., "Yes", "Blue bar").

# Respond ONLY with a valid JSON object matching this schema:
# {
#   "query_category": "Quantitative" | "Numerical" | "Reasoning",
#   "answer_category": "Pure Numerical" | "Alphanumeric/Units" | "Descriptive Text"
# }
# """

# output_dir = Path("./data/HuggingFaceM4/ChartQA/")
# output_dir.mkdir(parents=True, exist_ok=True)

# df_final_list = []

# for i, df_sample in enumerate(df_sample_train):
#     categorized_data = []
#     print(f"Sample {i}: sending {len(df_sample)} rows to the LLM...")

#     for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
#         query = str(row.get("query", row.get("question", "")))
#         answer = str(row.get("label", row.get("answer", "")))

#         try:
#             response = completion(
#                 model="deepseek/deepseek-chat",
#                 messages=[
#                     {"role": "system", "content": system_prompt},
#                     {"role": "user", "content": f"Query: {query}\nAnswer: {answer}"},
#                 ],
#                 response_format={"type": "json_object"},
#                 temperature=0.0,
#             )
#             categorized_data.append(json.loads(response.choices[0].message.content))
#         except Exception as e:
#             print(f"Error: {e}")
#             categorized_data.append({"query_category": "Unknown", "answer_category": "Unknown"})

#     df_res = pd.DataFrame(categorized_data)
#     df_final = pd.concat([df_sample.reset_index(drop=True), df_res], axis=1)
#     df_final_list.append(df_final)

#     output_path = output_dir / f"chartqa_categorized_sample_{i}.parquet"
#     df_final.to_parquet(output_path, index=False, engine="pyarrow")
#     print(f"Saved: {output_path}")

#     display(df_final.drop(columns=IMG_COLS, errors="ignore").head())

# df_final_0, df_final_1, df_final_2 = df_final_list

## Inferences, Observations 

In [ ]:
# Dart OCR

Chart data extraction

In [ ]:
import time
import torch
import pandas as pd
from transformers import (
    Pix2StructForConditionalGeneration, Pix2StructProcessor,
    DonutProcessor, VisionEncoderDecoderModel,
)

# --- config ---
N_TEST = 10                      # raise once happy
test_df = df_sample_train_0.head(N_TEST)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
# If you hit errors on "mps", set device = "cpu"
print("device:", device)

# --- DePlot ---
dp_proc = Pix2StructProcessor.from_pretrained("google/deplot")
dp_model = Pix2StructForConditionalGeneration.from_pretrained("google/deplot").to(device).eval()

def deplot_table(img):
    inputs = dp_proc(images=img, text="Generate underlying data table of the figure below:",
                     return_tensors="pt").to(device)
    with torch.no_grad():
        out = dp_model.generate(**inputs, max_new_tokens=512)
    return dp_proc.decode(out[0], skip_special_tokens=True).replace("<0x0A>", "\n")

# --- UniChart ---
uc_name = "ahmed-masry/unichart-base-960"
uc_proc = DonutProcessor.from_pretrained(uc_name)
uc_model = VisionEncoderDecoderModel.from_pretrained(uc_name).to(device).eval()

def unichart_table(img):
    prompt = "<extract_data_table> <s_answer>"
    decoder_ids = uc_proc.tokenizer(prompt, add_special_tokens=False, return_tensors="pt").input_ids
    pixel_values = uc_proc(img, return_tensors="pt").pixel_values
    with torch.no_grad():
        out = uc_model.generate(
            pixel_values.to(device),
            decoder_input_ids=decoder_ids.to(device),
            max_length=uc_model.decoder.config.max_position_embeddings,
            pad_token_id=uc_proc.tokenizer.pad_token_id,
            eos_token_id=uc_proc.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=4,
            return_dict_in_generate=True,
        )
    seq = uc_proc.tokenizer.batch_decode(out.sequences)[0]
    seq = seq.replace(uc_proc.tokenizer.eos_token, "").replace(uc_proc.tokenizer.pad_token, "")
    return seq.split("<s_answer>")[-1].strip()

# --- run on the test sample ---
def label_in(text, label):
    """Crude check: is the ground-truth answer present in the extracted table?"""
    return str(label[0]).lower() in text.lower()

results = []
for _, row in test_df.iterrows():
    img = decode_chart_image(row).convert("RGB")

    t0 = time.time(); dp = deplot_table(img);   t_dp = time.time() - t0
    t0 = time.time(); uc = unichart_table(img); t_uc = time.time() - t0

    results.append({
        "query": row["query"],
        "label": row["label"],
        "deplot_table": dp,
        "unichart_table": uc,
        "deplot_s": round(t_dp, 2),
        "unichart_s": round(t_uc, 2),
        "deplot_has_label": label_in(dp, row["label"]),
        "unichart_has_label": label_in(uc, row["label"]),
    })

df_ocr = pd.DataFrame(results)

# summary
print(df_ocr[["deplot_s", "unichart_s"]].mean().round(2).to_string())
print(df_ocr[["deplot_has_label", "unichart_has_label"]].mean().round(2).to_string())

# eyeball a few
for i in range(3):
    r = df_ocr.iloc[i]
    print(f"\n=== Q: {r['query']} | label: {r['label']} ===")
    print("--- DePlot ---\n", r["deplot_table"])
    print("--- UniChart ---\n", r["unichart_table"])

Attributte normalization

Jev ???

Jev v/s LLm tradeoffs